# Demo: `backtest()` with LLM-powered explanations

This notebook demonstrates the `ForecastingAssistant.backtest()` method
combined with the `ask()` method to get natural-language explanations
of backtesting results using a Google Gemini LLM.

The workflow:
1. Configure the assistant with a Google API key
2. Profile the data
3. Generate a forecasting plan
4. Create a cross-validation strategy
5. Run backtesting
6. Use `ask()` to get LLM-powered explanations of the results

**Requirements:** A valid Google API key with access to Gemini models.

In [15]:
import sys
sys.path.insert(0, "..")

In [16]:
import warnings

import numpy as np
import pandas as pd
from skforecast.model_selection import TimeSeriesFold

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")

## 1. Configure the Assistant with Google Gemini

Replace the placeholder below with your actual Google API key.

In [ ]:
GOOGLE_API_KEY = "your-google-api-key-here"  # Replace with your actual Google API key

In [18]:
assistant = ForecastingAssistant(
    llm="google:gemini-3-flash-preview",
    api_key=GOOGLE_API_KEY,
)

## 2. Synthetic Data

Create sample datasets for single-series and multi-series backtesting.

In [19]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series target with trend + seasonality
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

# Exogenous variables
promo = rng.normal(50, 10, n)
temperature = 15 + 10 * np.sin(2 * np.pi * np.arange(n) / 365) + rng.normal(0, 2, n)

# Single series with exogenous variables
df = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": promo,
    "temperature": temperature,
})

# Multi-series (long format)
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
    "promo": rng.normal(50, 10, n_multi * 3),
})

print(f"df:       {df.shape}")
print(f"df_multi: {df_multi.shape}")
display(df.head())

df:       (365, 4)
df_multi: (600, 4)


,date,sales,promo,temperature
0,2022-01-01,100.304717,64.814555,15.895627
1,2022-01-02,103.173890,42.564118,15.288682
2,2022-01-03,104.889824,41.777500,16.441694
3,2022-01-04,103.125168,52.023062,15.140855
4,2022-01-05,96.835295,58.443852,16.244312


---
## 3. Single Series Backtesting + LLM Explanation

Run backtesting on a single series with `ForecasterRecursive`, then use `ask()` to get an LLM-generated interpretation of the results.

### 3.1 Profile, Plan, and Backtest

In [20]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

CV explanation:
Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon.

CV: ============== 
TimeSeriesFold 
Initial train size    = 2022-09-12,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False



In [21]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
    prompt  = "Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon"
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

CV explanation:
The configuration uses the specific cutoff date requested for the initial training period. An expanding window approach is applied to utilize all available historical data as the evaluation progresses. Refitting is performed at every fold to ensure the model incorporates new observations prior to each 14-day forecast horizon, simulating a continuous retraining deployment cycle. Initial training up to 2022-09-12, expanding window, refit every fold, 14-step horizon.

CV: ============== 
TimeSeriesFold 
Initial train size    = 2022-09-12,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False



In [22]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
    prompt  = "I need to predict 14 steps every week. I get new data all sundays and want to use all my data for training."
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

CV explanation:
To match your weekly prediction cycle (every 7 days) with a 14-day horizon, I set fold_stride=7. I used the minimum viable initial_train_size of 236 to satisfy the model's high lag requirement (max_lag=118) while maximizing the number of evaluation folds. An expanding window (fixed_train_size=False) with refit=True ensures the model uses all available historical data as it 'arrives' each week. Using 65% of data (236 observations) for initial training, expanding window, refit every fold, 14-step horizon, 19 folds.

CV: ============== 
TimeSeriesFold 
Initial train size    = 236,
Steps                 = 14,
Fold stride           = 7,
Overlapping folds     = True,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False



In [23]:
result = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv,
    profile     = profile,
    plan        = plan,
)

print("Explanation:")
print(result.explanation)
print("\nMetrics:")
display(result.metrics)
print("\nPredictions (first rows):")
display(result.predictions.head(10))

100%|██████████| 19/19 [00:01<00:00, 13.65it/s]

Explanation:
Using 65% of data (236 observations) for initial training, expanding window, refit every fold, 14-step horizon, 19 folds. Results — mean_absolute_error: 2.0975, mean_squared_error: 7.3225, mean_absolute_scaled_error: 0.7155, mean_absolute_percentage_error: 0.0227.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.09747,7.32249,0.715532,0.022716



Predictions (first rows):


,fold,pred
2022-08-25,0,89.120598
2022-08-26,0,89.134445
2022-08-27,0,93.039311
2022-08-28,0,96.868049
2022-08-29,0,98.396155
2022-08-30,0,95.585254
2022-08-31,0,89.976783
2022-09-01,0,88.274187
2022-09-02,0,88.647170
2022-09-03,0,92.558469


In [24]:
print(result.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Time series cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 236,
    fold_stride        = 7,
    refit              = True,
    fixed

### 3.2 Ask the LLM about the backtesting results

Pass the `backtest_result` to `ask()` — the LLM receives metrics, predictions, and CV configuration in context.

In [25]:
answer = assistant.ask(
    prompt="Analyze the backtesting results. Are the metrics good? "
           "What does the error distribution suggest about the model?",
    backtest_result=result,
)
print(answer.explanation)

### Evaluation of Backtesting Metrics

The backtesting results indicate a highly accurate and reliable model for this sales series. By evaluating the specific metrics provided, we can determine the model's effectiveness relative to both the scale of the data and a baseline approach.

#### Benchmarking Against Naive Forecasts (MASE)
The most critical metric here is the **Mean Absolute Scaled Error (MASE) of 0.715**. In forecasting, a MASE below 1.0 means the model is performing better than a naive persistence forecast (simply predicting that today's value will be the same as yesterday's). A value of ~0.71 suggests that your model reduces error by approximately 28.5% compared to that basic baseline, indicating that the selected lags and exogenous variables (`promo`, `temperature`) are providing significant predictive power.

#### Precision and Scale (MAE & MAPE)
The **Mean Absolute Error (MAE) of 2.10** must be interpreted alongside the sales volume. With predicted values averaging aroun

In [26]:
answer = assistant.ask(
    prompt="Based on these backtesting results, what would you recommend "
           "to improve the forecast accuracy? Should I try different lags, "
           "add more features, or switch to a different model?",
    backtest_result=result,
)
print(answer.explanation)

Based on the backtesting results provided, your current model is performing quite well. A **MASE of 0.715** indicates that the model is significantly more accurate than a naive baseline, and a **MAPE of 2.27%** suggests very high precision relative to the scale of your sales data.

However, to further improve accuracy or ensure the model is robust, I recommend the following adjustments:

### 1. Re-evaluate Long-Term Lags
The current plan includes **lag 118** (approximately 4 months). In a dataset with only 365 observations, using such a high lag significantly reduces the number of training samples available to the model (it "consumes" nearly a third of your data).
*   **Recommendation**: Use the `calculate_lag_autocorrelation` tool to verify if lag 118 has a significant partial autocorrelation (PACF). If not, removing it will increase your training set size and may reduce overfitting.
*   **Alternative**: Focus on lags that capture weekly cycles (7, 14, 21) or monthly cycles (30), whic

---
## 4. Multi-Series Backtesting + LLM Explanation

Run backtesting on multiple series with `ForecasterRecursiveMultiSeries`.

In [27]:
profile_multi = assistant.profile(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
)

plan_multi = assistant.plan(
    profile    = profile_multi,
    steps      = 7,
    forecaster = "ForecasterRecursiveMultiSeries",
    estimator  = "LGBMRegressor",
)

cv_multi, _ = assistant.create_cv(
    profile = profile_multi,
    plan    = plan_multi,
)

result_multi = assistant.backtest(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    profile          = profile_multi,
    plan             = plan_multi,
)

print("Metrics (per series):")
display(result_multi.metrics)
print("\nPredictions (first rows):")
display(result_multi.predictions.head(10))

100%|██████████| 7/7 [00:00<00:00, 26.25it/s]

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,store_A,9.279120,368.934262,11.675822,0.096739
1,store_B,15.258092,715.127790,12.295175,0.073931
2,store_C,5.593974,73.459523,9.311378,0.043152
3,average,10.043729,385.840525,11.094125,0.071274
4,weighted_average,10.043729,385.840525,11.094125,0.071274
5,pooling,10.043729,385.840525,11.428570,0.071274



Predictions (first rows):


,level,fold,pred
2022-06-07,store_A,0,142.474526
2022-06-07,store_B,0,142.474526
2022-06-07,store_C,0,142.474526
2022-06-08,store_A,0,142.474526
2022-06-08,store_B,0,142.474526
2022-06-08,store_C,0,142.474526
2022-06-09,store_A,0,142.474526
2022-06-09,store_B,0,142.474526
2022-06-09,store_C,0,142.474526
2022-06-10,store_A,0,142.474526


In [28]:
answer_multi = assistant.ask(
    prompt="Compare the backtesting performance across the three stores. "
           "Which store is hardest to forecast and why might that be?",
    backtest_result=result_multi,
)
print(answer_multi.explanation)

Based on the backtesting metrics provided, here is an analysis of the forecasting performance across the three stores.

### Comparative Performance Analysis

The evaluation shows a clear hierarchy in how well the `ForecasterRecursiveMultiSeries` model performs for each store:

*   **Store C** is the most predictable. It has the lowest errors across all metrics, including the lowest **MAPE (4.32%)** and **MASE (9.31)**.
*   **Store A** shows moderate difficulty. While its absolute errors are lower than Store B's, it has the highest **MAPE (9.67%)**, meaning the errors are larger relative to its total revenue volume.
*   **Store B** is the most challenging series to forecast. It exhibits the highest **MAE (15.26)** and **MSE (715.13)**.

### Identifying the Hardest Store to Forecast

**Store B** is categorized as the most difficult to forecast for the following reasons:

1.  **Highest MASE (12.30):** The Mean Absolute Scaled Error (MASE) is the most reliable metric here because it is sca

---
## 5. General Forecasting Q&A (no data)

The `ask()` method can also answer general skforecast questions without any data context.

In [29]:
answer_general = assistant.ask(
    prompt="What is the difference between ForecasterRecursive and "
           "ForecasterDirect? When should I use each one?",
)
print(answer_general.explanation)

In **skforecast**, the choice between `ForecasterRecursive` and `ForecasterDirect` depends on your forecast horizon, the complexity of your data patterns, and your computational constraints.

### ForecasterRecursive

This is the standard approach for machine learning forecasting. It trains a **single model** that predicts one step ahead ($t+1$). To forecast further into the future, the model uses its own previous predictions as input features (lags) for the next step.

*   **How it works:** Predict $t+1$ $\rightarrow$ use $t+1$ to predict $t+2$ $\rightarrow$ use $t+2$ to predict $t+3$.
*   **Pros:**
    *   **Efficiency:** Only one model is trained and stored in memory.
    *   **Flexibility:** You can decide the number of steps to forecast (`steps`) at prediction time without retraining.
    *   **Speed:** Training is significantly faster than the Direct strategy.
*   **Cons:**
    *   **Error Propagation:** Small errors in the first prediction are fed back into the model, which can l

In [30]:
answer_intervals = assistant.ask(
    prompt="How do I add prediction intervals to my backtesting? "
           "What methods are available in skforecast?",
)
print(answer_intervals.explanation)

To add prediction intervals to your backtesting in **skforecast**, you use the `interval` argument within the backtesting functions. This allows the model to produce not just a point estimate, but also the upper and lower bounds of the predicted range.

### Available Methods

Skforecast provides three primary ways to generate these intervals, depending on the forecaster being used:

| Method | Description | Compatible Forecasters |
| :--- | :--- | :--- |
| **Bootstrapping** | Randomly samples from past residuals to simulate future paths. | `ForecasterRecursive`, `ForecasterDirect` |
| **Conformal Prediction** | A distribution-free method that provides mathematical guarantees on coverage. | All ML forecasters (including `Rnn` and `EquivalentDate`) |
| **Built-in (Parametric)** | Uses the mathematical assumptions of the model (e.g., Normal distribution). | `ForecasterStats` (ARIMA, ETS) |

---

### Adding Intervals to Backtesting

When calling `backtesting_forecaster` (or its multi-serie

---
## 6. Backtesting with Prediction Intervals + LLM Analysis

Run backtesting with prediction intervals and ask the LLM to evaluate coverage.

In [31]:
plan_intervals = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
    interval   = [10, 90],
)

cv_intervals, _ = assistant.create_cv(
    profile = profile,
    plan    = plan_intervals,
)

result_intervals = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv_intervals,
    profile     = profile,
    plan        = plan_intervals,
)

print("Metrics:")
display(result_intervals.metrics)
print("\nPredictions with intervals (first rows):")
display(result_intervals.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 13.44it/s]

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,3.184862,14.490499,1.082238,0.034049



Predictions with intervals (first rows):


,fold,pred,lower_bound,upper_bound
2022-09-13,0,93.641670,92.625056,94.129237
2022-09-14,0,90.002343,89.402665,90.755813
2022-09-15,0,85.876926,84.698623,87.088482
2022-09-16,0,83.741837,82.595850,84.918095
2022-09-17,0,87.837475,87.057970,88.509016
2022-09-18,0,91.933170,90.099689,92.510523
2022-09-19,0,94.988806,93.798783,95.820701
2022-09-20,0,93.758693,92.560896,94.963153
2022-09-21,0,89.148648,88.207854,90.248315
2022-09-22,0,84.682383,83.504080,85.901439


In [32]:
answer_coverage = assistant.ask(
    prompt="Evaluate the prediction intervals. Is the coverage close "
           "to the nominal 80%? Are the intervals too wide or too narrow?",
    backtest_result=result_intervals,
)
print(answer_coverage.explanation)

Based on the provided metrics and prediction samples, here is an evaluation of the prediction interval quality and calibration.

### Interval Calibration and Width

The prediction intervals appear to be **too narrow**, suggesting the model is overconfident. 

A key indicator is the relationship between the **Mean Absolute Error (MAE)** and the **Interval Width**:
*   The **MAE is 3.18**, meaning that, on average, the forecast misses the actual value by about 3.2 units.
*   The predicted **interval width** for the first step (2022-09-13) is only **1.50 units** ($94.13 - 92.63$). 
*   Even by the end of the horizon (2022-12-31), the width increases only to **2.80 units** ($92.52 - 89.72$).

If the average error is larger than the entire width of an "80% confidence" interval, the true values will likely fall outside that interval much more than 20% of the time. For a well-calibrated 80% interval, the width should typically be significantly larger than the MAE to account for the variance i

---
## 7. Shortcut: Fully Automatic Backtesting

The `backtest()` method can handle everything automatically — profiling, plan generation, and execution — with just the data and a `TimeSeriesFold`.

In [33]:
cv_simple = TimeSeriesFold(
    steps=14,
    initial_train_size=300,
    refit=False,
    fixed_train_size=False,
)

result_auto = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv_simple,
)

print(f"Auto-selected forecaster: {result_auto.plan.forecaster}")
print(f"Auto-selected estimator: {result_auto.plan.estimator}")
print(f"Uses exog: {result_auto.plan.use_exog}")
print("\nMetrics:")
display(result_auto.metrics)

100%|██████████| 5/5 [00:00<00:00, 794.80it/s]

Information of folds
--------------------
Number of observations used for initial training: 300
Number of observations used for backtesting: 65
    Number of folds: 5
    Number skipped folds: 0 
    Number of steps per fold: 14
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 9 observations.

Fold: 0
    Training:   0 -- 299  (n=300)
    Validation: 300 -- 313  (n=14)
Fold: 1
    Training:   No training in this fold
    Validation: 314 -- 327  (n=14)
Fold: 2
    Training:   No training in this fold
    Validation: 328 -- 341  (n=14)
Fold: 3
    Training:   No training in this fold
    Validation: 342 -- 355  (n=14)
Fold: 4
    Training:   No training in this fold
    Validation: 356 -- 364  (n=9)

Auto-selected forecaster: ForecasterRecursive
Auto-selected estimator: LGBMRegressor
Uses exog: True

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,4.110126,24.205572,1.41884,0.042661


In [34]:
answer_auto = assistant.ask(
    prompt="Summarize the full backtesting pipeline: what forecaster was chosen, "
           "what features were used, and how did it perform?",
    backtest_result=result_auto,
)
print(answer_auto.explanation)

This summary outlines the forecasting strategy and performance results based on the provided data and configuration.

### Selected Forecaster and Strategy
The pipeline uses a **ForecasterRecursive** paired with a **LightGBM (LGBMRegressor)** estimator. This strategy is designed for multi-step forecasting where the model predicts one day at a time, using its own previous predictions as inputs for subsequent steps until the 14-day horizon is reached. LightGBM was chosen for its efficiency and native ability to handle potential missing values in the dataset.

### Feature Engineering
The model captures complex temporal patterns using three distinct feature sets:
*   **Autoregressive Lags**: A combination of short-term lags (1, 3), weekly indicators (7, 8, 10, 12), and a long-term lag (118) to account for historical dependencies.
*   **Window Features**: Two rolling mean features were calculated—a 7-day window to capture weekly trends and a 91-day window to capture quarterly or seasonal mov